In [1]:
from stepmix.stepmix import StepMix
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pickle

In [ ]:
cluster_df = pd.read_csv("../data/master_data/2016_to_2023_clustering_data_set.csv")

len(cluster_df)

85013

In [7]:
cluster_cols = ["Age9",
                "Gend3",
                "Eth7",
                "Disab2_POP",
                "Educ6",
                "NSSEC5",
                "IMD10",
                "WorkStat8",
                "Child4",
                "HHLiv9",
                "Motivation_PC_Q",
                "motivd_POP"]

X = cluster_df[cluster_cols]

X.head()

,Age9,Gend3,Eth7,Disab2_POP,Educ6,NSSEC5,IMD10,WorkStat8,Child4,HHLiv9,Motivation_PC_Q,motivd_POP
0,0,0,3,1,0,0,1,1,1,2,1,3
1,2,0,3,1,0,1,3,1,0,1,3,3
2,2,1,0,1,0,0,6,0,0,0,2,3
3,2,0,1,1,0,0,3,0,1,4,4,3
4,2,1,1,0,0,0,3,1,1,4,0,2


In [8]:
results = []

for k in [25]:

    try:
    
        print(f"Fitting {k} classes...")

        model = StepMix(n_components=k,  measurement="categorical", random_state=42, n_init=50, max_iter=5000, abs_tol=1e-5)

        model.fit(X)

        with open(f"stepmix_new_{k}.pkl", "wb") as f:
             pickle.dump(model, f)

        posterior = model.predict_proba(X)
        predicted = model.predict(X)

        proportions = posterior.mean(axis=0)
        assignment = posterior.max(axis=1)

        loglik = model.score(X) * len(X)

        print(f"Log-likelihood : {loglik:,.2f}")
        print(f"AIC            : {model.aic(X):,.2f}")
        print(f"BIC            : {model.bic(X):,.2f}")
        print(f"Converged      : {model.converged_}")
        print(f"Iterations     : {model.n_iter_}")

        print("\nClass proportions")
        print(pd.Series(proportions).round(4))

        print("\nObserved class sizes")
        print(pd.Series(predicted).value_counts(normalize=True).sort_index().round(4))

        print("\nPosterior assignment certainty")
        print(pd.Series(assignment).describe())

        print(f"\nMean assignment probability : {assignment.mean():.4f}")
        print(f"Median assignment           : {np.median(assignment):.4f}")
        print(f"Minimum assignment          : {assignment.min():.4f}")

        temp = cluster_df.copy()
        temp["Class"] = predicted

        print("\nYear by latent class")
        print(pd.crosstab(temp["year"], temp["Class"], normalize="index").round(3))

        print("\nModel attributes")
        print(sorted(model.__dict__.keys()))

        params = model.get_parameters()

        print("\nParameter keys:")
        print(params.keys())

        measurement = params["measurement"]

        print("\nMeasurement parameters")
        print(type(measurement))

        if isinstance(measurement, dict):
            print("Measurement keys:")
            print(measurement.keys())

            for key, value in measurement.items():
                print(f"\n{key}")
                print(type(value))
                if hasattr(value, "shape"):
                    print("Shape:", value.shape)
        else:
            if hasattr(measurement, "shape"):
                print("Shape:", measurement.shape)
            else:
                print(measurement)

        results.append({"Classes": k,
                        "LogLik": loglik,
                        "AIC": model.aic(X),
                        "BIC": model.bic(X),
                        "Converged": model.converged_,
                        "Iterations": model.n_iter_,
                        "MeanAssignment": assignment.mean(),
                        "MedianAssignment": np.median(assignment),
                        "MinClass": proportions.min(),
                        "MaxClass": proportions.max(),
                        "EffectiveClasses": (proportions > 0.01).sum()})
        
    except Exception as e:
        results.append({"Classes": k, "Error": str(e)})

    pd.DataFrame(results).to_csv("stepmix_new_latent_class_analysis_results.csv", index=False)

Fitting 25 classes...
Fitting StepMix...


Initializations (n_init) : 100%|██████████| 50/50 [58:47<00:00, 70.55s/it, max_LL=-1.17e+6, max_avg_LL=-13.8] 


Log-likelihood : -1,170,299.57
AIC            : 2,343,647.14
BIC            : 2,357,897.39
Converged      : True
Iterations     : 308

Class proportions
0     0.0251
1     0.0668
2     0.0198
3     0.0221
4     0.0249
5     0.0176
6     0.0259
7     0.0884
8     0.0662
9     0.0188
10    0.0540
11    0.0398
12    0.0164
13    0.0385
14    0.0671
15    0.0064
16    0.0325
17    0.0193
18    0.0169
19    0.0205
20    0.0848
21    0.0339
22    0.0537
23    0.0386
24    0.1017
dtype: float64

Observed class sizes
0     0.0237
1     0.0674
2     0.0191
3     0.0214
4     0.0271
5     0.0156
6     0.0217
7     0.0952
8     0.0560
9     0.0181
10    0.0484
11    0.0390
12    0.0156
13    0.0405
14    0.0713
15    0.0060
16    0.0333
17    0.0188
18    0.0169
19    0.0194
20    0.0972
21    0.0318
22    0.0529
23    0.0362
24    0.1075
Name: proportion, dtype: float64

Posterior assignment certainty
count    85013.000000
mean         0.763193
std          0.198520
min          0.206210
25%    

In [9]:
results_df = pd.DataFrame(results)

print(results_df)

   Classes        LogLik           AIC           BIC  Converged  Iterations  \
0       25 -1.170300e+06  2.343647e+06  2.357897e+06       True         308   

   MeanAssignment  MedianAssignment  MinClass  MaxClass  EffectiveClasses  
0        0.763193          0.798164  0.006406  0.101689                24  
